# K-Means Clustering (Day 2 — Metrics and Evaluation)
## 23CSE301 ML Capstone · Review 2
### Person 2 | Track: K-Means Clustering
---
**Day 2 Scope:**
- Elbow curve to select best k
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index
- Comparison table across k values
- Re-fit final K-Means with chosen k

> All [STUDENT TASK] prompts require your original written analysis.


## 1. Setup (Same as Day 1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = os.path.join('..', 'data', 'ai4i2020.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'ai4i2020.csv')

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df['Type_encoded'] = df['Type'].map({'L': 0, 'M': 1, 'H': 2})

CLUSTER_FEATURES = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]',
    'Tool wear [min]', 'Type_encoded',
]

X_cluster = df[CLUSTER_FEATURES].copy()
cluster_scaler = StandardScaler()
X_scaled = cluster_scaler.fit_transform(X_cluster)
TRUE_LABELS = df['Machine failure'].values

print(f"Data ready. Shape: {X_scaled.shape}")


## 2. Elbow Curve

In [ ]:
# Compute inertia for k = 2 to 10
k_range = range(2, 11)
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                random_state=RANDOM_STATE)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    print(f"k={k}  inertia={km.inertia_:.1f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), inertias, 'o-', color='#2196F3', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_title('K-Means Elbow Curve')
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig('elbow_curve.png', bbox_inches='tight', dpi=150)
plt.show()


### Elbow Curve Observation
> **[STUDENT TASK]:** Look at the elbow curve plot above.  
> Where does the curve 'elbow' — i.e., where does inertia stop decreasing rapidly?  
> State the k value you will use going forward and why the elbow curve supports your choice.  
> Also consider: does this k align with the three product quality types (L/M/H)?


## 3. Evaluation Metrics for Multiple k Values

In [ ]:
# Compute all three metrics for k = 2 to 8
# NOTE: Silhouette is expensive — use a sample if dataset is large
# With n=10,000, full computation is acceptable

eval_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                random_state=RANDOM_STATE)
    km.fit(X_scaled)
    labels = km.labels_

    # Guard against degenerate clusters
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        print(f"k={k}: degenerate (only {n_unique} cluster) — skipping metrics")
        continue

    sil = silhouette_score(X_scaled, labels, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X_scaled, labels)
    ch  = calinski_harabasz_score(X_scaled, labels)

    eval_rows.append({
        'k': k,
        'Inertia': round(km.inertia_, 1),
        'Silhouette Score': round(sil, 4),        # Higher is better
        'Davies-Bouldin Index': round(db, 4),      # Lower is better
        'Calinski-Harabasz Index': round(ch, 1),  # Higher is better
    })
    print(f"k={k}  Sil={sil:.4f}  DB={db:.4f}  CH={ch:.1f}")

metrics_df = pd.DataFrame(eval_rows).set_index('k')
print("\n=== Clustering Metrics Across k Values ===")
metrics_df


### Metric Direction Reminder
- **Silhouette Score** → Higher is better (range: -1 to 1)
- **Davies-Bouldin Index** → **Lower is better**
- **Calinski-Harabasz Index** → Higher is better

> **[STUDENT TASK]:** From the table above, which k gives:
> - The highest Silhouette Score?
> - The lowest Davies-Bouldin Index?
> - The highest Calinski-Harabasz Index?
> Do they all agree? If not, how will you decide which k to use?  
> Write your reasoning here (3–4 sentences).


## 4. Final K-Means — Chosen k

In [ ]:
# === STUDENT: Update K_CHOSEN after inspecting the metrics table above ===
K_CHOSEN = 3  # Update this after examining the Elbow curve and metric table

km_final = KMeans(
    n_clusters=K_CHOSEN, init='k-means++',
    n_init=10, max_iter=300, random_state=RANDOM_STATE
)
km_final.fit(X_scaled)
labels_final = km_final.labels_

sil_final = silhouette_score(X_scaled, labels_final, random_state=RANDOM_STATE)
db_final  = davies_bouldin_score(X_scaled, labels_final)
ch_final  = calinski_harabasz_score(X_scaled, labels_final)

print(f"Final K-Means: k={K_CHOSEN}")
print(f"Silhouette Score:        {sil_final:.4f}  (higher is better)")
print(f"Davies-Bouldin Index:    {db_final:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index: {ch_final:.1f}  (higher is better)")
print(f"Cluster distribution: {dict(zip(*np.unique(labels_final, return_counts=True)))}")


---
## Day 2 Complete — Handoff Notes

**Next (Day 3):** PCA 2D scatter plot, t-SNE coordination, final notebook cleanup.

**Student must do before Day 3:**
1. Update K_CHOSEN in Cell 4 after examining the metrics.
2. Write the Elbow Curve observation (Section 2) and Metric observation (Section 3).
3. Make sure all cells run without errors.
4. Prepare to discuss WHY you chose this k during the viva.
